In [15]:
from collections import Counter 

In [16]:
class ITPparser:
    def __init__(self, filename):
        self.filename = filename
        self.atom_types = dict()
        self.nb_types = dict()
        self.bond_types = dict()
        self.pair_types = dict()
        self.angle_types = dict()
        self.dihedral_types = {'1': [], '2': [], '3': [], '4': [], '5': [], '8': [], '9': [], '10': [], '11': []}

        current_section = None
        with open(self.filename, "r") as f:
            for line in f:
                line = line.strip()
                if line.startswith('['):
                    current_section = line.strip('[]').strip()
                    continue
                if not line or line.startswith(';') or line.startswith('#'):
                    continue

                if current_section == 'atomtypes':
                    atom_type = parse_atomtypes(line)
                    self.atom_types[atom_type] = line

                elif current_section == 'nonbond_params':
                    nb_type = parse_nonbond(line)
                    self.nb_types[nb_type] = line

                elif current_section == 'bondtypes':
                    bond_aa = parse_bondtypes(line)
                    self.bond_types[bond_aa] = line

                elif current_section == 'pairtypes':
                    pair = parse_pairtypes(line)
                    self.pair_types[pair] = line

                elif current_section == 'angletypes':
                    angle_aa = parse_angletypes(line)
                    self.angle_types[angle_aa] = line

                elif current_section == 'dihedraltypes':
                    dihedral_aa, func, multi = parse_dihedraltypes(line)
                    self.dihedral_types[str(func)].append((dihedral_aa, multi, line))

def parse_atomtypes(line):
    """ Parses line from atomtypes section, returns str, AtomType """
    words = line.split()
    attype = words[0]

    return attype

def parse_nonbond(line):
    words = line.split()
    a1, a2 = words[:2]

    return (a1, a2)

def parse_bondtypes(line):
    words = line.split()
    a1, a2 = words[:2]

    return (a1, a2)

def parse_pairtypes(line):
    words = line.split()
    a1, a2 = words[:2]
    return (a1, a2)

def parse_angletypes(line):
    """
    Parses angletypes line. Returns str, str, str, AngleType, BondType/None
    """
    words = line.split()
    # theta = float(words[4])
    # k = (float(words[5]) / 2)

    return (words[0], words[1], words[2])

def parse_dihedraltypes(line):
    """ Parse dihedraltypes, returns (str,str,str,str), str, Type, bool """
    #print(line)
    words = line.split()
    a1, a2, a3, a4 = words[:4]
    func = words[4]
    
    if len(words) == 8:
        multi = words[7]
    else:
        multi = None

    return (a1, a2, a3, a4), func, multi



In [17]:
def find_missing_params(itp_new, itp_cphmd):
    '''Take in input ITPparser.attribute where each attribute is a dict of parameters.'''
    missing = []

    for param_type_new in list(itp_new.keys()):
        
        i = 0 # check match

        for param_type_cphmd in list(itp_cphmd.keys()):

            if Counter(param_type_new) == Counter(param_type_cphmd):

                i += 1
                break

        if i == 0:
            missing.append(param_type_new)
    
    return missing

In [18]:
def find_missing_angles(itp_new, itp_cphmd):
    '''Take in input ITPparser.attribute where each attribute is a dict of parameters.'''
    missing = []

    for param_type_new in list(itp_new.keys()):
        
        i = 0 # check match

        for param_type_cphmd in list(itp_cphmd.keys()):

            #if Counter(param_type_new) == Counter(param_type_cphmd):
            if param_type_new == param_type_cphmd:

                i += 1
                break

        if i == 0:
            missing.append(param_type_new)
    
    return missing

In [19]:
def find_missing_dihedrals(itp_new, itp_cphmd):
    '''Take in input ITPparser.dihedral_types.'''

    missing = {}

    for dihedral_func in list(itp_new.keys()):

        missing[dihedral_func] = []
        
        for dihedral_new in itp_new[dihedral_func]:
            i = 0 # check match
            
            for dihedral_cphmd in itp_cphmd[dihedral_func]:
                #if Counter(dihedral_new[0]) == Counter(dihedral_cphmd[0]):
                if dihedral_new[0] == dihedral_cphmd[0]:

                    i+= 1
                    break 

            if i == 0:
                missing[dihedral_func].append(dihedral_new)

    return missing

In [20]:
def insert_missing(charmm_ff, new_charmm_ff, missing_list, section_name):

    with open(charmm_ff, "r", encoding="utf-8") as fi:
        ff_lines = fi.readlines()

    # find the line index of the bondtypes header
    insert_at = None

    for idx, ln in enumerate(ff_lines):

        if "[" in ln and section_name in ln:

            while ff_lines[idx+1].startswith(";"):
                idx = idx + 1

            insert_at = idx + 1
            break

    to_add = [ln for ln in missing_list]

    if to_add:
        # ensure each line ends with a single newline
        new_lines = [ln.rstrip("\n") + "\n" for ln in to_add]
        ff_lines[insert_at:insert_at] = new_lines

        with open(new_charmm_ff, "w", encoding="utf-8") as fo:
            fo.writelines(ff_lines)


In [21]:
def insert_missing_dihedral(charmm_ff, new_charmm_ff, missing_list, section_name):

    with open(charmm_ff, "r", encoding="utf-8") as fi:
        ff_lines = fi.readlines()

    # find the line index of the bondtypes header
    insert_at = None

    for idx, ln in enumerate(ff_lines):

        if "[" in ln and section_name in ln:

            while ff_lines[idx+1].startswith(";") or ff_lines[idx+1].strip() == "":
                idx = idx + 1

            # print(ff_lines[idx+1].split()[4])
            # print(missing_list[0].split()[4])

            if ff_lines[idx+1].split()[4] == missing_list[0].split()[4]:
                insert_at = idx + 1
                break
            
            else:
                continue

    to_add = [ln for ln in missing_list]

    if to_add:
        # ensure each line ends with a single newline
        new_lines = [ln.rstrip("\n") + "\n" for ln in to_add]
        ff_lines[insert_at:insert_at] = new_lines

        with open(new_charmm_ff, "w", encoding="utf-8") as fo:
            fo.writelines(ff_lines)


In [22]:
# retrieve bond_types entries for each tuple in missing_bond (try both key orders)
def retrieve_missing_lines(missing_params, itp_new):
    '''Take in input ITPparser.attribute where each attribute is a dict of parameters.'''

    missing_lines = []
    missing_not_in_dict = []
    for key in missing_params:
        val = itp_new.get(key) or itp_new.get((key[1], key[0]))
        if val is None:
            missing_not_in_dict.append(key)
        else:
            missing_lines.append(val + "\n")
    return missing_lines


# Load ITP files

In [23]:
NEW_CHARMM_FF = "charmm36.itp"
CPHMD_BONDED = "charmm36-mar2019-cphmd.ff/ffbonded.itp.bck"
CPHMD_NONBONDED = "charmm36-mar2019-cphmd.ff/ffnonbonded.itp.bck"

In [24]:
itp_new = ITPparser(NEW_CHARMM_FF)
itp_cphmd_bonded = ITPparser(CPHMD_BONDED)
itp_cphmd_nonbonded = ITPparser(CPHMD_NONBONDED)

In [25]:
# ffbonded.itp

missing_bond = find_missing_params(itp_new.bond_types, itp_cphmd_bonded.bond_types)
missing_angle = find_missing_params(itp_new.angle_types, itp_cphmd_bonded.angle_types)
missing_dihedrals = find_missing_dihedrals(itp_new.dihedral_types, itp_cphmd_bonded.dihedral_types)

In [26]:
# ffnonbonded.itp

missing_nonbond = find_missing_params(itp_new.nb_types, itp_cphmd_nonbonded.nb_types)
missing_pair = find_missing_params(itp_new.pair_types, itp_cphmd_nonbonded.pair_types)

# Merge bonded parameters

In [27]:
if len(missing_bond) > 0:
    insert_missing(CPHMD_BONDED,
               "ffbonded_merged.itp", retrieve_missing_lines(missing_bond, itp_new.bond_types), "bondtypes")

if len(missing_angle) > 0:
    insert_missing("ffbonded_merged.itp",
               "ffbonded_merged.itp", retrieve_missing_lines(missing_angle, itp_new.angle_types), "angletypes")

if len(missing_dihedrals['2']) > 0:
    insert_missing_dihedral("ffbonded_merged.itp",
                   "ffbonded_merged.itp", [x[2] + '\n' for x in missing_dihedrals['2']], "dihedraltypes")

if len(missing_dihedrals['9']) > 0:
    insert_missing_dihedral("ffbonded_merged.itp",
                "ffbonded_merged.itp", [x[2] + '\n' for x in missing_dihedrals['9']], "dihedraltypes")

# Merge non bonded parameters

In [28]:
if len(missing_nonbond) > 0:    
    insert_missing(CPHMD_NONBONDED,
                    "ffnonbonded_merged.itp", retrieve_missing_lines(missing_nonbond, itp_new.nb_types), "nonbond_params")
    
if len(missing_pair) > 0:  
    insert_missing("ffnonbonded_merged.itp",
                "ffnonbonded_merged.itp", retrieve_missing_lines(missing_pair, itp_new.pair_types), "pairtypes")